# Import a CSV into `ig_market_data.db`

Loads one CSV written by `download_ig_prices.ipynb`
(`data/ig_<epic-slug>_<resolution-suffix>_<timestamp>.csv`) into the
per-resolution `candles_<suffix>` table of `data/ig_market_data.db` — the
same table and schema the download notebook's `SAVE_DB` path writes, created
here via `candle_db.init_candles_table()` and filled with `INSERT OR IGNORE`
keyed on `UNIQUE(epic, snapshot_time_utc)`, so re-running is idempotent.

The CSV holds only `snapshot_time_utc` + the flattened `{open,high,low,close}_{bid,ask,mid}_price`
and `last_traded_volume` columns. Two things it does **not** carry:

- **`epic` / `resolution`** — taken from `config.py` (`.env`), i.e. whatever
  `download_ig_prices.ipynb` was configured for when it wrote the CSV. Override
  `EPIC` / `RESOLUTION` in the config cell if you're importing a CSV from a
  different market or resolution.
- **the raw IG candle JSON blob** — reconstructed from the CSV's bid/ask
  columns so the DB's `data` column and flattened columns match a direct
  download. The exchange-local `snapshotTime` and per-node `lastTraded` aren't
  in the CSV, so they come back as absent / `null`.

Needs `python-dotenv` (for `config.py`); everything else is standard library.

In [1]:
%pip install -q python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 1. Config

`DB_PATH`, `EPIC`, `RESOLUTION` come from `config.py`'s `Config` (same as
`view_ig_prices.ipynb`). `CSV_PATH` defaults to the newest `data/ig_*.csv`;
set it explicitly to import a specific file.

In [2]:
import csv
import json
import sqlite3
from pathlib import Path

from candle_db import table_name_for_resolution
from config import Config

cfg = Config.from_env()

# epic + resolution are not in the CSV - take them from config.py / .env (what
# download_ig_prices.ipynb used to write it). Override if importing another market.
EPIC = cfg.epic
RESOLUTION = cfg.resolution
DB_PATH = cfg.db_path

# CSV to import: newest data/ig_*.csv by mtime. Set explicitly to pick another.
_candidates = sorted(Path("data").glob("ig_*.csv"), key=lambda p: p.stat().st_mtime)
assert _candidates, "No data/ig_*.csv found - run download_ig_prices.ipynb first, or set CSV_PATH."
CSV_PATH = _candidates[-1]

print(f"CSV_PATH   = {CSV_PATH}")
print(f"DB_PATH    = {DB_PATH}")
print(f"EPIC       = {EPIC!r}")
print(f"RESOLUTION = {RESOLUTION!r}  -> table {table_name_for_resolution(RESOLUTION)}")

CSV_PATH   = data\ig_nasdaq_daily_20260906204540.csv
DB_PATH    = data\ig_market_data.db
EPIC       = 'IX.D.NASDAQ.IFA.IP'
RESOLUTION = 'DAY'  -> table candles_1d


## 2. Read + check the CSV

Fail loudly unless the columns are exactly `constants/csv_headers.py`'s
`CSV_HEADERS` — this notebook only imports CSVs shaped by
`download_ig_prices.ipynb`.

In [3]:
from constants.csv_headers import CSV_HEADERS

with open(CSV_PATH, newline="") as fh:
    reader = csv.DictReader(fh)
    header = reader.fieldnames
    csv_rows = list(reader)

assert header == CSV_HEADERS, (
    f"Unexpected CSV columns.\n  got:      {header}\n  expected: {CSV_HEADERS}\n"
    "This notebook only imports CSVs written by download_ig_prices.ipynb."
)
print(f"{CSV_PATH.name}: {len(csv_rows)} data row(s), columns OK")
for row in csv_rows[:3]:
    print(dict(row))

ig_nasdaq_daily_20260906204540.csv: 213 data row(s), columns OK
{'snapshot_time_utc': '2025-12-30T13:00:00', 'open_bid_price': '25513.6', 'open_ask_price': '25515.6', 'open_mid_price': '25514.6', 'high_bid_price': '25578.3', 'high_ask_price': '25579.3', 'high_mid_price': '25578.8', 'low_bid_price': '25342.7', 'low_ask_price': '25344.7', 'low_mid_price': '25343.7', 'close_bid_price': '25417.3', 'close_ask_price': '25419.3', 'close_mid_price': '25418.3', 'last_traded_volume': '306075'}
{'snapshot_time_utc': '2025-12-31T13:00:00', 'open_bid_price': '25417.5', 'open_ask_price': '25419.5', 'open_mid_price': '25418.5', 'high_bid_price': '25504.6', 'high_ask_price': '25506.6', 'high_mid_price': '25505.6', 'low_bid_price': '25214.4', 'low_ask_price': '25216.4', 'low_mid_price': '25215.4', 'close_bid_price': '25218.2', 'close_ask_price': '25223.2', 'close_mid_price': '25220.7', 'last_traded_volume': '275806'}
{'snapshot_time_utc': '2026-01-01T13:00:00', 'open_bid_price': '25258.4', 'open_ask_pr

## 3. Import into `candles_<suffix>`

Each CSV row is rebuilt into a raw-IG-style candle dict, then flattened with
the producer's own `candle_csv.candle_to_row()` and inserted via
`candle_db.INSERT_COLUMNS` — identical to `download_ig_prices.ipynb`'s
`write_db()`, so imported rows are indistinguishable from downloaded ones
(bar the missing `snapshotTime` / `lastTraded` noted above).

In [4]:
from candle_csv import candle_to_row
from candle_db import INSERT_COLUMNS, init_candles_table
from constants.ig_candle_fields import CandleField, PriceField

LAST_TRADED = "lastTraded"  # raw-IG price-node key the CSV drops; stored as null


def _float(x):
    x = (x or "").strip()
    return float(x) if x else None


def _int(x):
    x = (x or "").strip()
    return int(x) if x else None


def csv_row_to_candle(r):
    """Rebuild a raw-IG-style candle dict from one CSV row so the DB's `data`
    blob and flattened columns match a direct download."""

    def node(prefix):
        return {
            PriceField.BID: _float(r[f"{prefix}_bid_price"]),
            PriceField.ASK: _float(r[f"{prefix}_ask_price"]),
            LAST_TRADED: None,
        }

    return {
        CandleField.SNAPSHOT_TIME_UTC: r["snapshot_time_utc"],
        CandleField.OPEN_PRICE: node("open"),
        CandleField.HIGH_PRICE: node("high"),
        CandleField.LOW_PRICE: node("low"),
        CandleField.CLOSE_PRICE: node("close"),
        CandleField.LAST_TRADED_VOLUME: _int(r["last_traded_volume"]),
    }


conn = sqlite3.connect(DB_PATH)
table = init_candles_table(conn, RESOLUTION)  # CREATE TABLE IF NOT EXISTS - producer's schema

insert_rows = []
for r in csv_rows:
    c = csv_row_to_candle(r)
    insert_rows.append((EPIC, RESOLUTION, *candle_to_row(c), json.dumps(c)))

before = conn.total_changes
placeholders = ", ".join("?" for _ in INSERT_COLUMNS)
conn.executemany(
    f"INSERT OR IGNORE INTO {table} ({', '.join(INSERT_COLUMNS)}) VALUES ({placeholders})",
    insert_rows,
)
conn.commit()
inserted = conn.total_changes - before
print(f"{DB_PATH}: +{inserted} new row(s) in `{table}` "
      f"({len(insert_rows) - inserted} already present, skipped by INSERT OR IGNORE)")

data\ig_market_data.db: +0 new row(s) in `candles_1d` (213 already present, skipped by INSERT OR IGNORE)


## 4. Verify

Read the rows back for this `epic` with `candle_db.load_candles` — the same
function `view_ig_prices.ipynb` uses.

In [5]:
from candle_db import load_candles

rows = list(load_candles(conn, RESOLUTION, epic=EPIC))
conn.close()

print(f"{len(rows)} candle(s) in `{table}` for {EPIC}")
if rows:
    print("first:", rows[0])
    print("last: ", rows[-1])

213 candle(s) in `candles_1d` for IX.D.NASDAQ.IFA.IP
first: {'epic': 'IX.D.NASDAQ.IFA.IP', 'resolution': 'DAY', 'snapshot_time_utc': '2025-12-30T13:00:00', 'open': 25514.6, 'high': 25578.8, 'low': 25343.7, 'close': 25418.3, 'open_mid_price': 25514.6, 'high_mid_price': 25578.8, 'low_mid_price': 25343.7, 'close_mid_price': 25418.3, 'volume': 306075}
last:  {'epic': 'IX.D.NASDAQ.IFA.IP', 'resolution': 'DAY', 'snapshot_time_utc': '2026-09-04T14:00:00', 'open': 29647.0, 'high': 29648.0, 'low': 29439.0, 'close': 29490.5, 'open_mid_price': 29647.0, 'high_mid_price': 29648.0, 'low_mid_price': 29439.0, 'close_mid_price': 29490.5, 'volume': 294097}
